# HW2.3: Analyzing Data Directly From S3

Reads the hourly `.parquet` files directly from `s3://dsan6000-<NetID>/wikipedia-hourly/` (no local `data/` folder involved), combines them into one `DataFrame`, and produces two seaborn line plots of hourly event counts.

In [ ]:
import boto3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
NETID = "jh2732"
BUCKET_NAME = f"dsan6000-{NETID}"
SOURCE_PREFIX = "wikipedia-hourly"


In [ ]:
s3_client = boto3.client("s3")
response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f"{SOURCE_PREFIX}/")
object_keys = [obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".parquet")]
print(f"Found {len(object_keys)} parquet files in s3://{BUCKET_NAME}/{SOURCE_PREFIX}/")
object_keys


In [ ]:
dataframes = [pd.read_parquet(f"s3://{BUCKET_NAME}/{key}") for key in object_keys]
events_df = pd.concat(dataframes, ignore_index=True)
events_df["hour"] = pd.to_datetime(events_df["file_ts_str"], format="%Y%m%d_%H%M%S")

print(events_df.shape)
events_df.head()


In [ ]:
hourly_totals = events_df.groupby("hour").size().reset_index(name="event_count")

plt.figure(figsize=(10, 5))
sns.lineplot(data=hourly_totals, x="hour", y="event_count", marker="o")
plt.title("Total Wikipedia Events per Hour")
plt.xlabel("Hour")
plt.ylabel("Event Count")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("images/hourly-events.svg")
plt.savefig("images/hourly-events.png")
plt.show()


In [ ]:
hourly_by_type = events_df.groupby(["hour", "type"]).size().reset_index(name="event_count")

plt.figure(figsize=(10, 5))
sns.lineplot(data=hourly_by_type, x="hour", y="event_count", hue="type", marker="o")
plt.title("Wikipedia Events per Hour by Type")
plt.xlabel("Hour")
plt.ylabel("Event Count")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("images/events-by-type.svg")
plt.savefig("images/events-by-type.png")
plt.show()
